In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col, sum, count, round, rand, when
import random
from datetime import datetime, timedelta

# 1. Configuración de nombres y listas para aleatoriedad
nombres_mascotas = ["Luna", "Simba", "Max", "Pelusa", "Rocky", "Bella", "Coco", "Milo", "Toby", "Kitty"]
especies = ["Perro", "Gato"]
razas = {"Perro": ["Labrador", "Beagle", "Boxer", "Poodle", "Bulldog"], 
         "Gato": ["Persa", "Angora", "Siamés", "Maine Coon", "Esfinge"]}
categorias = ["Alimento", "Aseo", "Juguetes", "Salud", "Accesorios"]

# 2. Generar Dimension de Mascotas (20 registros base)
data_mascotas = []
for i in range(1, 21):
    esp = random.choice(especies)
    data_mascotas.append(Row(
        id_mascota=f"M{i:03}",
        nombre=random.choice(nombres_mascotas),
        especie=esp,
        raza=random.choice(razas[esp]),
        fecha_nacimiento=str(datetime(2018, 1, 1) + timedelta(days=random.randint(0, 2000))).split()[0],
        id_cliente=f"C{random.randint(100, 110)}"
    ))

# 3. Generar Dimension de Productos (10 productos)
data_productos = [
    Row(id_producto=f"P{i:03}", 
        descripcion=f"Producto_{categorias[i%5]}_{i}", 
        categoria=categorias[i%5], 
        precio_unitario=round(random.uniform(5.0, 100.0), 2), 
        stock=random.randint(10, 100))
    for i in range(1, 11)
]

# 4. Generar Tabla de Hechos: Ventas (50 registros)
data_ventas = []
for i in range(1, 51):
    prod = random.choice(data_productos)
    cant = random.randint(1, 5)
    data_ventas.append(Row(
        id_venta=f"V{i:04}",
        fecha_venta=str(datetime(2026, 4, 1) + timedelta(days=random.randint(0, 25))).split()[0],
        id_mascota=random.choice(data_mascotas).id_mascota,
        id_producto=prod.id_producto,
        cantidad=cant,
        total_linea=round(prod.precio_unitario * cant, 2)
    ))

# Crear DataFrames y guardar como Delta Tables
df_mascotas = spark.createDataFrame(data_mascotas)
df_productos = spark.createDataFrame(data_productos)
df_ventas = spark.createDataFrame(data_ventas)

spark.sql("CREATE DATABASE IF NOT EXISTS mascotas_db")
df_mascotas.write.format("delta").mode("overwrite").saveAsTable("mascotas_db.dim_mascotas")
df_productos.write.format("delta").mode("overwrite").saveAsTable("mascotas_db.dim_productos")
df_ventas.write.format("delta").mode("overwrite").saveAsTable("mascotas_db.fact_ventas")

print("Estructura de 50 registros creada en 'mascotas_db'")

In [0]:
# Reporte: Ventas totales por Categoría y Especie
reporte_completo = spark.table("mascotas_db.fact_ventas") \
    .join(spark.table("mascotas_db.dim_mascotas"), "id_mascota") \
    .join(spark.table("mascotas_db.dim_productos"), "id_producto")

final = reporte_completo.groupBy("categoria", "especie") \
    .agg(
        sum("cantidad").alias("unidades_vendidas"),
        round(sum("total_linea"), 2).alias("ingresos_totales"),
        count("id_venta").alias("n_transacciones")
    ) \
    .orderBy("ingresos_totales", ascending=False)

display(final)